# RAG in 60 Seconds: Give Your LLM a Memory

From the AI60secLab video: https://www.youtube.com/watch?v=E9vvmCZp38U

RAG means Retrieval-Augmented Generation. Before the model answers, you first retrieve the right facts, then hand them to the model with the question. This notebook shows the retrieval half, which is the part that makes answers trustworthy.

Runs in Google Colab with nothing to install.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Our clinic is open Monday to Friday, 8 am to 5 pm.",
    "Appointments can be rescheduled up to 24 hours before the visit.",
    "We accept Visa, Mastercard, and bank transfer.",
    "The office is closed on public holidays.",
    "New patients should arrive 15 minutes early with an ID.",
]

vectorizer = TfidfVectorizer()
doc_vectors = vectorizer.fit_transform(documents)
print("Indexed", len(documents), "documents")

In [ ]:
def retrieve(question, top_k=2):
    q_vec = vectorizer.transform([question])
    scores = cosine_similarity(q_vec, doc_vectors)[0]
    best = scores.argsort()[::-1][:top_k]
    return [(documents[i], round(float(scores[i]), 3)) for i in best]

for doc, score in retrieve("Can I pay by card?"):
    print(score, doc)

In [ ]:
# This is the prompt a real RAG app would send to the LLM:
question = "Can I pay by card?"
context = retrieve(question, top_k=1)[0][0]

prompt = f"""Answer using only this context:
Context: {context}
Question: {question}"""
print(prompt)

## What to notice
The model never guesses. It answers from documents you control, which is why RAG is the pattern behind chatbots that answer from your own files.